In [ ]:
import warnings
warnings.filterwarnings(action="ignore", message=r"datetime.datetime.utcnow")

# PERSIAPAN

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Pada laporan praktikum, sisipkan penjelasan singkat mengenai library `sklearn` (https://scikit-learn.org/stable/) di **BAB 2** pada bagian metode yang sesuai. Gunakan *website* dokumentasi resminya sebagai referensi.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

drive_path = 'drive/MyDrive/sml'

if os.path.exists(drive_path):
    print(f"The directory '{drive_path}' exists.")
else:
    print(f"The directory '{drive_path}' does not exist.")

The directory 'drive/MyDrive/sml' exists.


In [ ]:
df_full = pd.read_csv("Park_Visitors.csv")
df_full.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Park_Visitors.csv'

Deskripsi:
* **VisitorID**: pengenal unik untuk setiap pengunjung
* **Gender**: jenis kelamin pengunjung (0="Female" atau 1="Male")
* **Age**: usia pengunjung (dalam tahun)
* **Time Spent Hours**: durasi rata-rata per kunjungan untuk pengunjung ini (dalam jam)
* **Distance Km**: jarak dari rumah pengunjung ke taman (dalam kilometer)
* **Is Weekend**: apakah pengunjung ini biasanya berkunjung pada akhir pekan ("Yes" = akhir pekan paling sering; "No" = hari kerja paling sering)
* **Arrival Session**: waktu kedatangan yang paling sering ("Morning", "Afternoon", atau "Evening")
* **Exhibit Main**: zona bertema yang paling sering dikunjungi di taman ("Art", "Science", "History", atau "Mixed")
* **Spend Total**: rata-rata pengeluaran per kunjungan di stan makanan & toko merchandise (dalam ribu rupiah; target regresi)


Melihat struktur data mentah

In [ ]:
df_full.info()

Pemilihan variabel

In [ ]:
df = df_full.copy()
df = df_full.drop(columns=['VisitorID'], axis=1)
df.head()

Mempersingkat nama kolom

In [ ]:
df.columns = [
    'gender',
    'age',
    'time_spent',
    'distance',
    'is_weekend',
    'arrival_session',
    'exhibit_main',
    'spend_total'
    ]
df.head()

Mengubah data kategorik yang terbaca numerik oleh Python

In [ ]:
df = df.copy()
for c in ["gender"]:
    df[c] = df[c].fillna(df[c].mode()[0]).astype(int).astype("object")

# EXPLORATORY DATA ANALYSIS

### Ringkasan

In [ ]:
df.info()

In [ ]:
df.describe(include = 'number')

In [ ]:
df.describe(include = ['category', 'object'])

### Scatter Plot

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 3))
plt.scatter(df['time_spent'], df['spend_total'], color='blue', alpha=0.5)
plt.title('Sebaran Time Spent vs Spend Total')
plt.xlabel('Time Spent (jam)')
plt.ylabel('Spend Total (ribu rupiah)')
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(5, 3))
df.boxplot(column='spend_total', by='arrival_session', grid=False)
plt.title('Sebaran Spend Total menurut Arrival Session')
plt.suptitle('')
plt.xlabel('Arrival Session')
plt.ylabel('Spend Total (jam)')
plt.show()

# PREPROCESSING

### Missing Value

In [ ]:
print('Total Missing Value')
print(df.isnull().sum())

In [ ]:
df_lengkap = df.copy()

Jika terdapat missing value pada data, maka terdapat dua opsi untuk menanganinya:


1. Imputasi pada sel data yang kosong dengan suatu nilai (misal: mean/median/modus) pada kolom yang mengandung missing value

In [ ]:
# # 1. Contoh penanganan missing value menggunakan rata-rata
col_with_mis_val = ['time_spent', 'age', 'distance', 'spend_total']
for col in col_with_mis_val:
  df_lengkap[col] = df_lengkap[col].fillna(df_lengkap[col].mean())

2. Menghapus baris yang mengandung missing value

In [ ]:
# 2. Contoh penanganan missing value dengan menghapus baris yang mengandung missing value
df_lengkap = df_lengkap.dropna(axis=0)

In [ ]:
print('Total Missing Value')
print(df_lengkap.isnull().sum())

### Outlier

In [ ]:
def cek_outlier(df, kolom=None):
  num_cols = df.select_dtypes(include=[np.number]).columns
  if kolom is None:
    kolom = num_cols
  else:
    kolom = [c for c in kolom if c in num_cols]  # filter khusus numerik

  df_checked = df[kolom]
  df_checked.plot(
    kind="box",
    subplots=True,
    layout=(1, len(kolom)),
    figsize=(len(kolom) * 3, 5),
    sharey=False
  )

  plt.suptitle("Cek Outlier Semua Kolom")
  plt.show()

In [ ]:
cek_outlier(df_lengkap)

Hapus outlier (hanya gunakan jika diperlukan)

In [ ]:
def hapus_outlier(df, kolom=None, max_iter=1):
    num_cols = df.select_dtypes(include=[np.number]).columns
    if kolom is None:
        kolom = num_cols
    else:
        kolom = [c for c in kolom if c in num_cols]  # hanya numerik

    df_clean = df.copy()
    for i in range(max_iter):
        before = df_clean.shape[0]

        mask = pd.Series(True, index=df_clean.index)
        for c in kolom:
            Q1, Q3 = df_clean[c].quantile([0.25, 0.75])
            IQR = Q3 - Q1
            mask &= df_clean[c].between(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

        df_clean = df_clean[mask]
        after = df_clean.shape[0]

        if before == after:  # berhenti kalau tidak ada perubahan
            break
        print(f'Iterasi {i+1}: {before} -> {after} sampel')

    return df_clean

In [ ]:
df_bersih = df_lengkap.copy()
df_bersih = hapus_outlier(df_bersih, max_iter=10)
df_bersih.info()

In [ ]:
cek_outlier(df_bersih)

### Data Duplikat

In [ ]:
print(df_bersih.duplicated().sum(), 'baris terduplikat')
display(df_bersih[df_bersih.duplicated()].head())

Hapus data yang terduplikat

In [ ]:
df_siap = df_bersih.copy()
df_siap.drop_duplicates(inplace=True)
df_siap.info()

# SPLITTING DATA & TRANSFORMASI

### Splitting Data

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# Pisahkan fitur (X) dan target (y)
col_y = 'spend_total'
X = df_siap.drop(columns=col_y)
y = df_siap[col_y]

print('Variabel prediktor:', ', '.join(X.columns.values))
print('Variabel respons/target:', col_y)

In [ ]:
# Split train (80%) dan test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)

print("Ukuran data train:", X_train.shape[0], 'baris')
print("Ukuran data test :", X_test.shape[0], 'baris')

### One-Hot Encoding (untuk kategorikal)

In [ ]:
cat_cols = ['gender', 'is_weekend', 'arrival_session', 'exhibit_main']

X_train_dum = pd.get_dummies(X_train, columns=cat_cols, drop_first=True, dtype='uint8')
X_test_dum = pd.get_dummies(X_test, columns=cat_cols, drop_first=True, dtype='uint8')
X_test_dum = X_test_dum.reindex(columns=X_train_dum.columns, fill_value=0)
X_train_dum.info()

### Standardisasi (untuk numerik)

In [ ]:
from sklearn.preprocessing import StandardScaler

num_cols = ['age', 'time_spent', 'distance']

scaler = StandardScaler()
scaler.fit(X_train_dum[num_cols]) # fit HANYA di data train

X_train_std = X_train_dum.copy()
X_test_std  = X_test_dum.copy()
X_train_std[num_cols] = scaler.transform(X_train_dum[num_cols])
X_test_std[num_cols]  = scaler.transform(X_test_dum[num_cols])

In [ ]:
stats_train = pd.DataFrame({
    "mean_before": X_train_dum[num_cols].mean(),
    "std_before":  X_train_dum[num_cols].std(ddof=0),
    "mean_after":  X_train_std[num_cols].mean(),
    "std_after":   X_train_std[num_cols].std(ddof=0),
}).round(3)

# Cek hasil scaling pada data training
stats_train

# REGRESI

In [ ]:
# @title
def unstd(coef_table, stats_train, intercept="(Intercept)"):
    mu, s = stats_train["mean_before"], stats_train["std_before"]
    coefs = coef_table.set_index("variable")["coefficient_std"]
    # res = pd.Series(np.nan, index=coefs.index) # default NaN
    res = coefs.copy()
    num = coefs.index.intersection(mu.index) # kolom numerik

    res.loc[num] = coefs.loc[num] / s.loc[num] # slope numerik
    res.loc[intercept] = coefs.get(intercept, 0) - (coefs.reindex(mu.index).fillna(0) * (mu / s)).sum()

    return coef_table.assign(coefficient=coef_table["variable"].map(res))

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

### Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
# Membuat model
model_lin = LinearRegression()
# Training model
model_lin.fit(X_train_dum, y_train)

In [ ]:
# Estimasi parameter
coef_lin = pd.DataFrame({
    "variable": ["(Intercept)"] + list(X_train_dum.columns),
    "coefficient": [model_lin.intercept_, *model_lin.coef_]
})
coef_lin

In [ ]:
# Testing dan evaluasi
y_pred_lin = model_lin.predict(X_test_dum)

mse_lin  = mean_squared_error(y_test, y_pred_lin)
rmse_lin = np.sqrt(mse_lin)
mape_lin = mean_absolute_percentage_error(y_test, y_pred_lin)

print(f"MSE  (Linear): {mse_lin:.4f}")
print(f"RMSE (Linear): {rmse_lin:.4f}")
print(f"MAPE (Linear): {mape_lin:.4%}")

In [ ]:
# Prediksi
pred_lin = pd.DataFrame({"y": y_test, "y_pred": y_pred_lin}, index=y_test.index)
pred_lin["error"] = pred_lin["y"] - pred_lin["y_pred"]
pred_lin.head()

## Penalized Regression

### Ridge Regression

#### Tanpa tuning parameter

In [ ]:
from sklearn.linear_model import Ridge

# Membuat model
model_rdg = Ridge(alpha=0.1)
# Training model
model_rdg.fit(X_train_std, y_train)

#### Dengan tuning parameter

In [ ]:
from sklearn.linear_model import RidgeCV

# Kandidat lambda untuk tuning parameter
lambdas_rdg = [0.01, 0.1, 1, 10, 100]

# Training dan cross-validation (pilih lambda dengan MSE terkecil)
model_rdg = RidgeCV(alphas=lambdas_rdg, store_cv_results=True)
model_rdg.fit(X_train_std, y_train)

In [ ]:
# Menghitung MSE untuk setiap lambda yang digunakan
plt.figure(figsize=(6, 3))
plt.plot(lambdas_rdg, model_rdg.cv_results_.mean(axis=0), marker='o')
plt.xscale('log')
plt.axvline(model_rdg.alpha_, linestyle='--', color='orange', label=f'best λ = {model_rdg.alpha_}')
plt.xlabel('lambda')
plt.ylabel('CV MSE')
plt.title('Tuning Parameter Lambda vs MSE Regresi Ridge')

ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Lambda terbaik
print("Best Lambda:", model_rdg.alpha_)
print("Best Cross-Validation Score:", model_rdg.best_score_)

#### Estimasi, evaluasi, prediksi

In [ ]:
# Estimasi parameter
coef_rdg = pd.DataFrame({
    "variable": ["(Intercept)"] + list(X_train_std.columns),
    "coefficient_std": [model_rdg.intercept_, *model_rdg.coef_]
})
coef_rdg = unstd(coef_rdg, stats_train)
coef_rdg

In [ ]:
# Testing dan evaluasi
y_pred_rdg = model_rdg.predict(X_test_std)

mse_rdg  = mean_squared_error(y_test, y_pred_rdg)
rmse_rdg = np.sqrt(mse_rdg)
mape_rdg = mean_absolute_percentage_error(y_test, y_pred_rdg)

print(f"MSE  (Ridge): {mse_rdg:.4f}")
print(f"RMSE (Ridge): {rmse_rdg:.4f}")
print(f"MAPE (Ridge): {mape_rdg:.4%}")

In [ ]:
# Prediksi
pred_rdg = X_test_std.copy()
pred_rdg['y'] = y_test
pred_rdg['y_pred'] = y_pred_rdg
pred_rdg["error"] = pred_rdg["y"] - pred_rdg["y_pred"]
pred_rdg.head()

### LASSO Regression

#### Tanpa tuning parameter

In [ ]:
from sklearn.linear_model import Lasso

# Membuat model
model_lso = Lasso(alpha=0.1)
# Training model
model_lso.fit(X_train_std, y_train)

#### Dengan tuning parameter

In [ ]:
from sklearn.linear_model import LassoCV

# Kandidat lambda untuk tuning parameter
lambdas_lso = [1]

# Training dan cross-validation (pilih lambda dengan MSE terkecil)
model_lso = LassoCV(alphas=lambdas_lso, cv=5, max_iter=10000)  # max_iter dinaikkan agar konvergen
model_lso.fit(X_train_std, y_train)

In [ ]:
# Menghitung MSE untuk setiap lambda yang digunakan
plt.figure(figsize=(6, 3))
plt.plot(model_lso.alphas_, model_lso.mse_path_.mean(axis=1), marker='o')
plt.axvline(model_lso.alpha_, linestyle='--', color='orange', label=f'best λ = {model_lso.alpha_}')
plt.xscale('log')
plt.xlabel('lambda')
plt.ylabel('CV MSE')
plt.title('Tuning Parameter Lambda vs MSE Regresi LASSO')

ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Lambda terbaik
print("Lambda terbaik:", model_lso.alpha_)

#### Estimasi, evaluasi, prediksi

In [ ]:
# Estimasi parameter
coef_lso = pd.DataFrame({
    "variable": ["(Intercept)"] + list(X_train_std.columns),
    "coefficient_std": [model_lso.intercept_, *model_lso.coef_]
})
coef_lso = unstd(coef_lso, stats_train)
coef_lso

In [ ]:
# Testing dan evaluasi
y_pred_lso = model_lso.predict(X_test_std)

mse_lso  = mean_squared_error(y_test, y_pred_lso)
rmse_lso = np.sqrt(mse_lso)
mape_lso = mean_absolute_percentage_error(y_test, y_pred_lso)

print(f"MSE  (Lasso): {mse_lso:.4f}")
print(f"RMSE (Lasso): {rmse_lso:.4f}")
print(f"MAPE (Lasso): {mape_lso:.4%}")

In [ ]:
# Prediksi
pred_lso = X_test_std.copy()
pred_lso['y'] = y_test
pred_lso['y_pred'] = y_pred_lso
pred_lso["error"] = pred_lso["y"] - pred_lso["y_pred"]
pred_lso.head()

### Elastic Net Regression

#### Tanpa tuning parameter

In [ ]:
from sklearn.linear_model import ElasticNet

# Membuat model
model_eln = ElasticNet(alpha=10, l1_ratio=0.1)
# Training model
model_eln.fit(X_train_std, y_train)

#### Dengan tuning parameter

In [ ]:
from sklearn.linear_model import ElasticNetCV

# Kandidat lambda dan l1_ratio untuk tuning parameter
lambdas_eln = [0.001, 0.01, 0.1, 1, 10, 100]
l1_ratios_eln = [0.1, 0.5, 0.7, 0.9, 1.0]

# Training dan cross-validation (pilih lambda dengan MSE terkecil)
model_eln = ElasticNetCV(alphas=lambdas_eln, l1_ratio=l1_ratios_eln, cv=5, max_iter=10000)
model_eln.fit(X_train_std, y_train)

In [ ]:
# Menghitung MSE untuk setiap lambda dan l1_ratio yang digunakan
plt.figure(figsize=(6, 4))

for i, l1 in enumerate(np.atleast_1d(model_eln.l1_ratio)):
    alphas_i = model_eln.alphas_[i] if model_eln.alphas_.ndim == 2 else model_eln.alphas_
    mse_i = model_eln.mse_path_[i].mean(axis=1)
    plt.plot(alphas_i, mse_i, marker='o', label=f'l1_ratio = {l1}')

plt.axvline(model_eln.alpha_, linestyle='--', color='gray', label=f'best λ = {model_eln.alpha_}')
plt.xscale('log')
plt.xlabel('lambda')
plt.ylabel('CV MSE')
plt.title('Tuning Parameter Lambda dan L1 Ratio vs MSE Regresi Elastic Net')

ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Lambda dan l1 ratio terbaik
print("Lambda terbaik:", model_eln.alpha_)
print("L1 ratio terbaik:", model_eln.l1_ratio_)

#### Estimasi, evaluasi, prediksi

In [ ]:
# Estimasi parameter
coef_eln = pd.DataFrame({
    "variable": ["(Intercept)"] + list(X_train_std.columns),
    "coefficient_std": [model_eln.intercept_, *model_eln.coef_]
})
coef_eln = unstd(coef_eln, stats_train)
coef_eln

In [ ]:
# Testing dan evaluasi
y_pred_eln = model_eln.predict(X_test_std)

mse_eln  = mean_squared_error(y_test, y_pred_eln)
rmse_eln = np.sqrt(mse_eln)
mape_eln = mean_absolute_percentage_error(y_test, y_pred_eln)

print(f"MSE  (ElasticNet): {mse_eln:.4f}")
print(f"RMSE (ElasticNet): {rmse_eln:.4f}")
print(f"MAPE (ElasticNet): {mape_eln:.4%}")

In [ ]:
# Prediksi
pred_eln = X_test_std.copy()
pred_eln['y'] = y_test
pred_eln['y_pred'] = y_pred_eln
pred_eln["error"] = pred_eln["y"] - pred_eln["y_pred"]
pred_eln.head()

### Perbandingan Koefisien Linear vs Penalized Regression

In [ ]:
coef_all = pd.DataFrame({
    "Variabel": ["(Intercept)"] + list(X_train_dum.columns),
    "Linear": coef_lin['coefficient'],
    "Ridge": coef_rdg['coefficient'],
    "Lasso": coef_lso['coefficient'],
    "ElasticNet": coef_eln['coefficient'],
})
coef_all

## Tree-based Regression

### Regression Tree

In [ ]:
from sklearn.tree import DecisionTreeRegressor

#### Tanpa tuning parameter

In [ ]:
# Membuat model
model_tre = DecisionTreeRegressor(max_depth=2)
# Training model
model_tre.fit(X_train_dum, y_train)

#### Dengan tuning parameter

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
# Membuat model untuk di-tuning
model_tre_ = DecisionTreeRegressor()

# Menentukan parameter grid untuk GridSearchCV
param_grid_tre = {
    'max_depth': list(range(1, 20)),
    'min_samples_split': list(range(2,60))
}

In [ ]:
# Melakukan grid search dengan cross validation
grid_search_tre = GridSearchCV(estimator=model_tre_, param_grid=param_grid_tre,
                           cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
grid_search_tre.fit(X_train_dum, y_train)

# Menampilkan parameter terbaik
print("Best Parameters:", grid_search_tre.best_params_)
print("Best Cross-Validation Score", -grid_search_tre.best_score_)

In [ ]:
# Model terbaik
model_tre = grid_search_tre.best_estimator_
model_tre

#### Feature importances

Hasil dari feature importances dapat digunakan untuk menyeleksi variabel/fitur yang paling dianggap tidak penting. Hasil dari seleksi fitur dapat meningkatkan keakuratan, namun juga dapat menurunkan.

In [ ]:
importances = model_tre.feature_importances_
fi_tre = pd.DataFrame({
    'Feature': X_train_dum.columns,
    'Importance': importances
})
fi_tre.sort_values(by='Importance', ascending=False)

Variabel-variabel yang dianggap paling tidak penting dapat ditinggal apabila ingin dilakukan analisis ulang.

#### Visualisasi, evaluasi, prediksi

In [ ]:
from sklearn.tree import plot_tree

In [ ]:
# Visualisasi
plt.figure(figsize=(8, 4), dpi=300)
plot_tree(model_tre, feature_names = X_train_dum.columns, filled=True)
plt.title('Pohon Regresi')
plt.show()

In [ ]:
# Testing dan evaluasi
y_pred_tre = model_tre.predict(X_test_dum)

mse_tre  = mean_squared_error(y_test, y_pred_tre)
rmse_tre = np.sqrt(mse_tre)
mape_tre = mean_absolute_percentage_error(y_test, y_pred_tre)

print(f"MSE  (Regression Tree): {mse_tre:.4f}")
print(f"RMSE (Regression Tree): {rmse_tre:.4f}")
print(f"MAPE (Regression Tree): {mape_tre:.4%}")

In [ ]:
# Prediksi
pred_tre = X_test_dum.copy()
pred_tre['y'] = y_test
pred_tre['y_pred'] = y_pred_tre
pred_tre["error"] = pred_tre["y"] - pred_tre["y_pred"]
pred_tre.head()

### Random Forest Regression

In [ ]:
from sklearn.ensemble import RandomForestRegressor

#### Tanpa tuning parameter

In [ ]:
# Membuat model
model_rfo = RandomForestRegressor(random_state=5, max_depth=3)
# Training model
model_rfo.fit(X_train_dum, y_train)

#### Dengan tuning parameter

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
# Membuat model untuk di-tuning
model_rfo_ = RandomForestRegressor(random_state=5, max_depth=3)

# Menentukan parameter grid untuk GridSearchCV
param_grid_rfo = {
    'n_estimators': [1, 5, 10, 20] # jumlah pohon
}

In [ ]:
# Melakukan grid search dengan cross validation
grid_search_rfo = GridSearchCV(estimator=model_rfo_, param_grid=param_grid_rfo, cv=5,
                           scoring='neg_mean_squared_error', n_jobs=-1)
grid_search_rfo.fit(X_train_dum, y_train)

# Menampilkan parameter terbaik
print("Best Parameters:", grid_search_rfo.best_params_)
print("Best Cross-Validation Score", -grid_search_rfo.best_score_)

In [ ]:
# Model terbaik
model_rfo = grid_search_rfo.best_estimator_
model_rfo

#### Feature importances

In [ ]:
importances = model_rfo.feature_importances_
fi_rfo = pd.DataFrame({
    'Feature': X_train_dum.columns,
    'Importance': importances
})
fi_rfo.sort_values(by='Importance', ascending=False)

#### Visualisasi, evaluasi, prediksi

In [ ]:
from sklearn.tree import plot_tree

In [ ]:
# Visualisasi (salah satu pohon dari model random forest)
plt.figure(figsize=(8, 4), dpi=300)
plot_tree(model_rfo.estimators_[0], feature_names = X_train_dum.columns, filled=True)
plt.title('Pohon Regresi')
plt.show()

In [ ]:
# Testing dan evaluasi
y_pred_rfo = model_rfo.predict(X_test_dum)

mse_rfo  = mean_squared_error(y_test, y_pred_rfo)
rmse_rfo = np.sqrt(mse_rfo)
mape_rfo = mean_absolute_percentage_error(y_test, y_pred_rfo)

print(f"MSE  (Random Forest): {mse_rfo:.4f}")
print(f"RMSE (Random Forest): {rmse_rfo:.4f}")
print(f"MAPE (Random Forest): {mape_rfo:.4%}")

In [ ]:
# Prediksi
pred_rfo = X_test_dum.copy()
pred_rfo['y'] = y_test
pred_rfo['y_pred'] = y_pred_rfo
pred_rfo["error"] = pred_rfo["y"] - pred_rfo["y_pred"]
pred_rfo.head()

## Support Vector Regression (SVR)

In [ ]:
from sklearn.svm import SVR

#### Tanpa tuning parameter

In [ ]:
# Membuat model
model_svr = SVR(kernel='rbf', epsilon=0.1)
# Melatih model
model_svr.fit(X_train_dum, y_train)

#### Dengan tuning parameter

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
model_svr_ = SVR()
param_grid_svr = {
    'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
    'epsilon': [0.01, 0.1, 1]
}

In [ ]:
# Melakukan grid search dengan cross validation
grid_search_svr = GridSearchCV(estimator=model_svr_, param_grid=param_grid_svr, cv=5,
                           scoring='neg_mean_squared_error', n_jobs=-1)
grid_search_svr.fit(X_train_dum, y_train)

# Menampilkan parameter terbaik
print("Best Parameters:", grid_search_svr.best_params_)
print("Best Cross-Validation Score", -grid_search_svr.best_score_)

In [ ]:
# Model terbaik
model_svr = grid_search_svr.best_estimator_
model_svr

#### Evaluasi dan prediksi

In [ ]:
# Testing dan evaluasi
y_pred_svr = model_svr.predict(X_test_dum)

mse_svr  = mean_squared_error(y_test, y_pred_svr)
rmse_svr = np.sqrt(mse_svr)
mape_svr = mean_absolute_percentage_error(y_test, y_pred_svr)

print(f"MSE  (SVR): {mse_svr:.4f}")
print(f"RMSE (SVR): {rmse_svr:.4f}")
print(f"MAPE (SVR): {mape_svr:.4%}")

In [ ]:
# Prediksi
pred_svr = X_test_dum.copy()
pred_svr['y'] = y_test
pred_svr['y_pred'] = y_pred_svr
pred_svr["error"] = pred_svr["y"] - pred_svr["y_pred"]
pred_svr.head()

# PERBANDINGAN METRIK EVALUASI

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge Regression', 'Lasso Regression', 'ElasticNet Regression',
              'Regression Tree', 'Random Forest Regression', 'Support Vector Regression'],
    'MSE': [mse_lin, mse_rdg, mse_lso, mse_eln, mse_tre, mse_rfo, mse_svr],
    'RMSE': [rmse_lin, rmse_rdg, rmse_lso, rmse_eln, rmse_tre, rmse_rfo, rmse_svr],
    'MAPE': [mape_lin, mape_rdg, mape_lso, mape_eln, mape_tre, mape_rfo, mape_svr],
})
comparison['MAPE (%)'] = comparison['MAPE'] * 100
comparison = comparison.sort_values(by='MAPE (%)')
comparison